In [19]:
import os
import re
import wave
import subprocess
import tempfile
from pathlib import Path

from tqdm.notebook import tqdm

DATA_DIR = Path("../data")


def _sorted_timestamps(directory: Path, suffix: str) -> list[tuple[float, Path]]:
    files = sorted(directory.glob(f"*{suffix}"), key=lambda p: float(p.stem))
    return [(float(p.stem), p) for p in files]


def _write_image_concat(frames: list[tuple[float, Path]], tmp_dir: str, fps_fallback: float = 25.0) -> str:
    concat_path = os.path.join(tmp_dir, "images.txt")
    with open(concat_path, "w", encoding="utf-8") as f:
        for i, (ts, path) in enumerate(frames):
            dur = (frames[i + 1][0] - ts) if (i + 1 < len(frames)) else (1.0 / fps_fallback)
            if dur <= 0:
                dur = 1.0 / fps_fallback
            f.write(f"file '{path.resolve()}'\n")
            f.write(f"duration {dur:.9f}\n")
        f.write(f"file '{frames[-1][1].resolve()}'\n")
    return concat_path


def _merge_audio_wave(chunks: list[tuple[float, Path]], tmp_dir: str) -> tuple[str, wave._wave_params]:
    out_path = os.path.join(tmp_dir, "merged.wav")
    first_params = None
    with wave.open(out_path, "wb") as out:
        for i, (_, path) in enumerate(tqdm(chunks, desc="  merging audio", leave=False)):
            with wave.open(str(path), "rb") as wf:
                if i == 0:
                    first_params = wf.getparams()
                    out.setparams(first_params)
                else:
                    p = wf.getparams()
                    # hard fail if anything important differs (prevents “ffmpeg makes it obvious” situations)
                    if (p.nchannels != first_params.nchannels or
                        p.sampwidth != first_params.sampwidth or
                        p.framerate != first_params.framerate or
                        p.comptype  != first_params.comptype):
                        raise ValueError(f"Audio params mismatch:\n  first={first_params}\n  this ={p}\n  file ={path}")
                out.writeframes(wf.readframes(wf.getnframes()))
    return out_path, first_params


def _run_ffmpeg_with_progress(cmd: list[str], total_frames: int, desc: str) -> tuple[int, str]:
    progress_cmd = cmd + ["-progress", "pipe:1", "-nostats"]
    bar = tqdm(total=total_frames, desc=desc, unit="frame", leave=False)
    last_frame = 0
    stderr_text = ""

    with subprocess.Popen(
        progress_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        universal_newlines=True,
    ) as proc:
        for line in proc.stdout:
            m = re.match(r"frame=(\d+)", line.strip())
            if m:
                cur = int(m.group(1))
                if cur > last_frame:
                    bar.update(cur - last_frame)
                    last_frame = cur

        stderr_text = proc.stderr.read()
        proc.wait()

    if total_frames > last_frame:
        bar.update(total_frames - last_frame)
    bar.close()
    return proc.returncode, stderr_text


def process_run(run_dir: Path) -> None:
    images_dir = run_dir / "images"
    audio_dir = run_dir / "audio"
    output_dir = run_dir / "processed"
    output_dir.mkdir(exist_ok=True)
    output_path = output_dir / "0.mp4"

    frames = _sorted_timestamps(images_dir, ".png")
    if not frames:
        print(f"[{run_dir.name}] No images found, skipping.")
        return

    chunks = _sorted_timestamps(audio_dir, ".wav")

    t0_images = frames[0][0]
    t0_audio = chunks[0][0] if chunks else t0_images
    t0 = min(t0_images, t0_audio)

    frames = [(ts - t0, p) for ts, p in frames]
    audio_offset = t0_audio - t0  # seconds; >0 => audio starts later

    with tempfile.TemporaryDirectory() as tmp_dir:
        concat_path = _write_image_concat(frames, tmp_dir)

        cmd = [
            "ffmpeg", "-y",
            "-hide_banner",
            # make timestamps sane (prevents negative-ts weirdness popping/clicking with offsets)
            "-fflags", "+genpts",
            "-avoid_negative_ts", "make_zero",
            "-f", "concat", "-safe", "0", "-i", concat_path,
        ]

        map_args = ["-map", "0:v"]
        filter_args = []

        if chunks:
            merged_wav, params = _merge_audio_wave(chunks, tmp_dir)
            cmd += ["-i", merged_wav]

            # apply offset WITHOUT resampling/async “fixups” (those often cause crackles)
            delay_ms = int(round(max(audio_offset, 0.0) * 1000.0))
            trim_s = max(-audio_offset, 0.0)

            a = []
            if trim_s > 0:
                a.append(f"atrim=start={trim_s:.6f}")
            a.append("asetpts=PTS-STARTPTS")
            if delay_ms > 0:
                a.append(f"adelay={delay_ms}|{delay_ms}")

            if a:
                filter_args = ["-af", ",".join(a)]

            map_args += ["-map", "1:a"]

        tmp_output = os.path.join(tmp_dir, "output.mp4")

        cmd += map_args + filter_args + [
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            "-fps_mode", "vfr",
            "-c:a", "alac",
            "-shortest",
            tmp_output,
        ]

        returncode, stderr = _run_ffmpeg_with_progress(
            cmd,
            total_frames=len(frames),
            desc=f"  encoding [{run_dir.name}]",
        )

        if returncode != 0:
            print(f"[{run_dir.name}] ffmpeg failed:\n{stderr}")
        else:
            import shutil
            shutil.copy2(tmp_output, str(output_path))
            print(f"[{run_dir.name}] Written to {output_path}")


run_dirs = sorted(
    [d for d in DATA_DIR.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)

for run_dir in tqdm(run_dirs, desc="runs"):
    process_run(run_dir)

runs:   0%|          | 0/1 [00:00<?, ?it/s]

  merging audio:   0%|          | 0/5587 [00:00<?, ?it/s]

  encoding [0]:   0%|          | 0/2381 [00:00<?, ?frame/s]

[0] Written to ../data/0/processed/0.mp4


In [17]:
from pathlib import Path
import wave

run_dirs = sorted(
    [d for d in DATA_DIR.iterdir() if d.is_dir() and d.name.isdigit()],
    key=lambda d: int(d.name),
)

for run_dir in run_dirs:
    chunks = sorted((run_dir / 'audio').glob('*.wav'), key=lambda p: float(p.stem))
    if not chunks:
        print(f'[{run_dir.name}] No audio chunks, skipping.')
        continue

    out_path = run_dir / 'processed' / '0.wav'
    out_path.parent.mkdir(exist_ok=True)

    with wave.open(str(out_path), 'wb') as out:
        for i, path in enumerate(chunks):
            with wave.open(str(path), 'rb') as wf:
                if i == 0:
                    out.setparams(wf.getparams())
                out.writeframes(wf.readframes(wf.getnframes()))

    print(f'[{run_dir.name}] Written to {out_path}')

[0] Written to ../data/0/processed/0.wav
